In [1]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
dataset.keys()

dict_keys(['train', 'test', 'unsupervised'])

In [8]:
dataset['train'].features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [5]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

# Splitting the Dataset

```
Original training set
        │
        ├───────────────┐
        ↓               ↓
     Train          Validation

Original test set
        ↓
      Test
      

In [13]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
})


In [14]:
dataset = DatasetDict({
    "train": split_dataset['train'],
    "validation":split_dataset['test'],
    "test": dataset['test']
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
})


# Map function

In [15]:
# The Hugging Face Dataset.map() method applies a function to the dataset.
# we will use this later
def add_text_length(example):
    example["text_length"] = len(example["text"])
    return example

In [16]:
small_dataset = dataset["train"].select(range(5))

small_dataset = small_dataset.map(
    add_text_length
)

print(small_dataset)

Map: 100%|██████████| 5/5 [00:00<00:00, 230.28 examples/s]

Dataset({
    features: ['text', 'label', 'text_length'],
    num_rows: 5
})


In [17]:
print(small_dataset[0])

{'text': 'Every generation fully believes it is living in the end times. This has been true for thousands of years now. And movies like this feed on this. How did they get the great Orson Welles to narrate this train wreck? This is a documentary about the biblical prophecies of Armageddon. It tries to link the prophecies as well as it can to what was happening in the times it was made, making it obviously dated and kind of silly.<br /><br />The reenactments look like they are out of "Unsolved Mysteries" but without the high production values. People should have been embarrassed to take part in this.<br /><br />In short, the movie is dated, silly, reactionary, and useless. Good if you want a good laugh, but not good enough to actually look for.', 'label': 0, 'text_length': 743}


#  Batched Tokenization

In [18]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print(tokenizer)

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [19]:
# example
text = dataset["train"][0]["text"]

tokens = tokenizer(
    text,
    truncation=True,
    max_length=256
)

print(tokens)

{'input_ids': [101, 2296, 4245, 3929, 7164, 2009, 2003, 2542, 1999, 1996, 2203, 2335, 1012, 2023, 2038, 2042, 2995, 2005, 5190, 1997, 2086, 2085, 1012, 1998, 5691, 2066, 2023, 5438, 2006, 2023, 1012, 2129, 2106, 2027, 2131, 1996, 2307, 25026, 23447, 2000, 6583, 11335, 2618, 2023, 3345, 12006, 1029, 2023, 2003, 1037, 4516, 2055, 1996, 10213, 17678, 5369, 9243, 1997, 2849, 18655, 5280, 1012, 2009, 5363, 2000, 4957, 1996, 17678, 5369, 9243, 2004, 2092, 2004, 2009, 2064, 2000, 2054, 2001, 6230, 1999, 1996, 2335, 2009, 2001, 2081, 1010, 2437, 2009, 5525, 6052, 1998, 2785, 1997, 10021, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 2128, 8189, 6593, 8163, 2298, 2066, 2027, 2024, 2041, 1997, 1000, 4895, 19454, 7178, 15572, 1000, 2021, 2302, 1996, 2152, 2537, 5300, 1012, 2111, 2323, 2031, 2042, 10339, 2000, 2202, 2112, 1999, 2023, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1999, 2460, 1010, 1996, 3185, 2003, 6052, 1010, 10021, 1010, 4668, 5649, 1010, 1998, 11809, 1012, 

In [22]:
# Tokenization Function 

def tokenization_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256
    )

In [23]:
# Tokenize the training dataset

tokenized_train = dataset['train'].map(
    tokenization_function,
    batched=True
)

print(tokenized_train)

Map: 100%|██████████| 22500/22500 [00:06<00:00, 3551.06 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 22500
})


In [25]:
tokenized_validation = dataset["validation"].map(
    tokenization_function,
    batched=True
)

print(tokenized_validation)

Map: 100%|██████████| 2500/2500 [00:00<00:00, 3618.36 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2500
})


In [26]:
tokenized_test = dataset["test"].map(
    tokenization_function,
    batched=True
)

print(tokenized_test)

Map: 100%|██████████| 25000/25000 [00:05<00:00, 4777.64 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})


In [27]:
tokenized_dataset = DatasetDict({
    "train": tokenized_train,
    "validation": tokenized_validation,
    "test": tokenized_test
})

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 22500
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})


In [40]:
tokenized_dataset["train"]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 22500
})

In [31]:
# convert IDs to tokens 
example = tokenized_dataset["train"][0]

print(example["input_ids"])

print()

tokens = tokenizer.convert_ids_to_tokens(
    example["input_ids"]
)

print(tokens)

[101, 2296, 4245, 3929, 7164, 2009, 2003, 2542, 1999, 1996, 2203, 2335, 1012, 2023, 2038, 2042, 2995, 2005, 5190, 1997, 2086, 2085, 1012, 1998, 5691, 2066, 2023, 5438, 2006, 2023, 1012, 2129, 2106, 2027, 2131, 1996, 2307, 25026, 23447, 2000, 6583, 11335, 2618, 2023, 3345, 12006, 1029, 2023, 2003, 1037, 4516, 2055, 1996, 10213, 17678, 5369, 9243, 1997, 2849, 18655, 5280, 1012, 2009, 5363, 2000, 4957, 1996, 17678, 5369, 9243, 2004, 2092, 2004, 2009, 2064, 2000, 2054, 2001, 6230, 1999, 1996, 2335, 2009, 2001, 2081, 1010, 2437, 2009, 5525, 6052, 1998, 2785, 1997, 10021, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 2128, 8189, 6593, 8163, 2298, 2066, 2027, 2024, 2041, 1997, 1000, 4895, 19454, 7178, 15572, 1000, 2021, 2302, 1996, 2152, 2537, 5300, 1012, 2111, 2323, 2031, 2042, 10339, 2000, 2202, 2112, 1999, 2023, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1999, 2460, 1010, 1996, 3185, 2003, 6052, 1010, 10021, 1010, 4668, 5649, 1010, 1998, 11809, 1012, 2204, 2065, 20

In [35]:
decoded_text = tokenizer.decode(
    example["input_ids"]
)

print(decoded_text)

[CLS] every generation fully believes it is living in the end times. this has been true for thousands of years now. and movies like this feed on this. how did they get the great orson welles to narrate this train wreck? this is a documentary about the biblical prophecies of armageddon. it tries to link the prophecies as well as it can to what was happening in the times it was made, making it obviously dated and kind of silly. < br / > < br / > the reenactments look like they are out of " unsolved mysteries " but without the high production values. people should have been embarrassed to take part in this. < br / > < br / > in short, the movie is dated, silly, reactionary, and useless. good if you want a good laugh, but not good enough to actually look for. [SEP]


# Padding

In [36]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(data_collator)

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')


In [42]:
# Create a separate dataset for testing the DataCollator
collator_dataset = tokenized_dataset.remove_columns(["text"])

print(collator_dataset)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 22500
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
})


In [43]:
features = [
    collator_dataset["train"][i]
    for i in range(3)
]

print("Original sequence lengths:")

for i, feature in enumerate(features):
    print(
        f"Example {i}: "
        f"{len(feature['input_ids'])} tokens"
    )


batch = data_collator(features)

print("\nBatch input shape:")
print(batch["input_ids"].shape)

print("\nBatch attention-mask shape:")
print(batch["attention_mask"].shape)

print("\nBatch labels:")
print(batch["labels"])

Original sequence lengths:
Example 0: 180 tokens
Example 1: 141 tokens
Example 2: 256 tokens

Batch input shape:
torch.Size([3, 256])

Batch attention-mask shape:
torch.Size([3, 256])

Batch labels:
tensor([0, 1, 0])


# Complete Version of pipeline

In [44]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorWithPadding


# 1. Load dataset
dataset = load_dataset("stanfordnlp/imdb")


# 2. Train / validation / test split
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": dataset["test"]
})


# 3. Load tokenizer
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)


# 4. Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )


# 5. Tokenize dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


# 6. Separate dataset for collator demonstration
collator_dataset = tokenized_dataset.remove_columns(
    ["text"]
)


# 7. Dynamic padding collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# 8. Test with 3 examples
features = [
    collator_dataset["train"][i]
    for i in range(3)
]

batch = data_collator(features)


# 9. Inspect result
print("Batch input shape:")
print(batch["input_ids"].shape)

print("\nBatch attention-mask shape:")
print(batch["attention_mask"].shape)

print("\nBatch labels:")
print(batch["labels"])

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yuvan\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 25000/25000 [00:04<00:00, 5128.38 examples/s]

Batch input shape:
torch.Size([3, 256])

Batch attention-mask shape:
torch.Size([3, 256])

Batch labels:
tensor([0, 1, 1])
